# 04. Propensity to Buy + Кластеризация

**Цель**
- Построить модель склонности к покупке (propensity to convert)
- Сегментировать клиентов (K-Means) **только на train**, чтобы избежать утечки
- Проанализировать кластеры по conversion и uplift

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Загрузка данных

In [ ]:
data = joblib.load(PROJECT_ROOT / "outputs" / "prepared_data.joblib")
uplift_test = np.load(PROJECT_ROOT / "outputs" / "uplift_test.npy")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
t_test = data["t_test"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Propensity to Buy (склонность к покупке)

In [ ]:
propensity_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)
propensity_model.fit(X_train, y_train)

prop_train = propensity_model.predict_proba(X_train)[:, 1]
prop_test = propensity_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, prop_test)
print(f"Propensity ROC-AUC (test): {auc:.4f}")
print(f"Mean propensity test: {prop_test.mean():.4f}")

In [ ]:
joblib.dump(propensity_model, PROJECT_ROOT / "models" / "propensity_to_buy.joblib")
print("Saved models/propensity_to_buy.joblib")

## 3. Кластеризация (только на train)

In [ ]:
N_CLUSTERS = 5

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(X_train)

clusters_train = kmeans.predict(X_train)
clusters_test = kmeans.predict(X_test)

print("Cluster sizes (train):")
print(pd.Series(clusters_train).value_counts().sort_index())

In [ ]:
# Профиль кластеров на test (conversion + uplift)
profile = pd.DataFrame({
    "cluster": clusters_test,
    "conversion": y_test,
    "treatment": t_test,
    "uplift_score": uplift_test,
    "propensity": prop_test,
})

cluster_stats = profile.groupby("cluster").agg(
    n=("conversion", "size"),
    conversion_rate=("conversion", "mean"),
    mean_uplift=("uplift_score", "mean"),
    mean_propensity=("propensity", "mean"),
).round(4)

print(cluster_stats.sort_values("mean_uplift", ascending=False))

In [ ]:
# PCA для визуализации
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_test)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=clusters_test, cmap="tab10", alpha=0.5, s=12
)
plt.colorbar(scatter, label="Cluster")
plt.title("Customer clusters (PCA of test set)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_clusters_pca.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cluster_stats["conversion_rate"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Conversion rate by cluster")
axes[0].set_ylabel("Conversion")
axes[0].tick_params(axis="x", rotation=0)

cluster_stats["mean_uplift"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Mean uplift score by cluster")
axes[1].set_ylabel("Uplift")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_cluster_profiles.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Сохранение

In [ ]:
joblib.dump(kmeans, PROJECT_ROOT / "models" / "kmeans.joblib")

cluster_stats.to_csv(PROJECT_ROOT / "outputs" / "cluster_profiles.csv")

profile.to_csv(PROJECT_ROOT / "outputs" / "test_with_clusters.csv", index=False)

print("Saved:")
print(" - models/kmeans.joblib")
print(" - outputs/cluster_profiles.csv")
print(" - outputs/test_with_clusters.csv")

## Выводы

- Propensity-модель даёт baseline-вероятность покупки (полезно для сравнения с uplift).
- Кластеры обучены **только на train** → нет утечки.
- Кластеры с высоким `mean_uplift` — приоритетные сегменты для коммуникации.
- Кластеры с высокой conversion, но низким uplift — «sure things» (покупают и без предложения).